In [1]:
import os
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition\\prototype'

In [2]:
os.chdir('../')
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [3]:
from datetime import datetime, timezone
from typing import List, Dict
import requests
from include.common import read_yaml
from include.constants import CONFIG_FILE_PATH
import json

In [4]:
CONFIG_FILE_PATH

WindowsPath('config/config.yaml')

# Reading in API

In [5]:
USGS_URL = read_yaml(CONFIG_FILE_PATH, verbose=False).data_ingestion.source_URL
USGS_URL

'https://earthquake.usgs.gov/fdsnws/event/1/query'

In [6]:
USGS_URL = read_yaml(CONFIG_FILE_PATH, verbose=False).data_ingestion.source_URL

def fetch_raw_usgs_window(
    start_time: datetime,
    end_time: datetime,
    min_magnitude: float = 2.5,
) -> List[Dict]:
    """
    Fetch one time window of earthquake data from USGS and return
    a flattened list of dictionaries.
    """
    params = {
        "format": "geojson",
        "starttime": start_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "endtime": end_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "minmagnitude": min_magnitude,
        "orderby": "time-asc",
        "limit": 20000,
    }

    response = requests.get(USGS_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    return data

In [7]:
time = datetime(2026, 1, 1, tzinfo=timezone.utc)
print(time.strftime("%Y-%m-%dT%H:%M:%S"))

end_time = time.replace(hour=23, minute=59, second=0, microsecond=0)
print(end_time.strftime("%Y-%m-%dT%H:%M:%S"))

2026-01-01T00:00:00
2026-01-01T23:59:00


In [9]:
start_time = datetime(2026, 1, 1, tzinfo=timezone.utc)
end_time = start_time.replace(hour=23, minute=59, second=0, microsecond=0)

# get start and end times for today
# start_time = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
# end_time = datetime.now(timezone.utc)

data = fetch_raw_usgs_window(
    start_time=start_time,
    end_time=end_time,
    min_magnitude=2.5,
)

os.makedirs("prototype/samples", exist_ok=True)
output_path = f"prototype/samples/usgs_{start_time.strftime('%Y_%m_%d')}_to_{end_time.strftime('%Y_%m_%d')}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Exported JSON to: {output_path}")

Exported JSON to: prototype/samples/usgs_2026_01_01_to_2026_01_01.json


# Checking Idempotency

In [8]:
# Test API idempotency by calling the same request multiple times
# and comparing normalized responses (ignoring volatile metadata fields).

def normalize_usgs_response(resp: dict) -> dict:
    normalized = {
        "type": resp.get("type"),
        "bbox": resp.get("bbox"),
        "features": resp.get("features", []),
        "metadata": dict(resp.get("metadata", {})),
    }
    # Remove fields that can change between calls even for same query
    normalized["metadata"].pop("generated", None)

    # Ensure deterministic order before comparison
    normalized["features"] = sorted(
        normalized["features"],
        key=lambda x: (
            x.get("id", ""),
            (x.get("properties") or {}).get("time", -1),
        ),
    )
    return normalized


num_trials = 9
responses = [
    fetch_raw_usgs_window(start_time=start_time, end_time=end_time, min_magnitude=2.5)
    for _ in range(num_trials)
]

normalized = [normalize_usgs_response(r) for r in responses]
baseline = json.dumps(normalized[0], sort_keys=True)

matches = [json.dumps(r, sort_keys=True) == baseline for r in normalized]

print(f"Trials: {num_trials}")
print(f"All responses idempotent (normalized): {all(matches)}")
for i, is_same in enumerate(matches, start=1):
    print(f"  Trial {i}: {'MATCH' if is_same else 'DIFF'}")

Trials: 9
All responses idempotent (normalized): True
  Trial 1: MATCH
  Trial 2: MATCH
  Trial 3: MATCH
  Trial 4: MATCH
  Trial 5: MATCH
  Trial 6: MATCH
  Trial 7: MATCH
  Trial 8: MATCH
  Trial 9: MATCH


# Delete `/sample` folder

In [19]:
def delete_files_in_samples(folder_path: str = "prototype/samples") -> int:
    """
    Delete all files directly inside `folder_path` if it exists.
    Returns the number of deleted files.
    """
    if not os.path.isdir(folder_path):
        return 0

    deleted_count = 0
    for entry in os.scandir(folder_path):
        if entry.is_file() or entry.is_symlink():
            os.remove(entry.path)
            deleted_count += 1

    return deleted_count

deleted_files = delete_files_in_samples()
print(f"Deleted {deleted_files} files from path")

Deleted 1 files from path
